# 8단계: 단어 인식 방식 정밀 시험 + 소리 모델이 잡을 수 있는 위급 소리

07 결과: Whisper-small이 "도와주세요"를 34번 중 32번 알아들었고, 연속 3시간(C)에서 단어가 인식된 건 2번(시간당 0.67회)뿐이었습니다. 소리 모델만으로는 재현율 90%에서 시간당 100회 이상이라 쓸 수 없었습니다.
이번에는 **단어 인식 방식이 실제로 쓸 만한지** 더 정밀하게 확인합니다. 학습은 하지 않고 평가만 하며, **GPU(T4) 런타임을 권장**합니다. 20~40분 걸립니다.

1. **단어 규칙 보강**: "연구해서" 속의 "구해"를 위급 단어로 오인한 문제를 고쳤습니다. (단어 앞이 띄어쓰기·문장 시작일 때만 인정)
2. **Whisper 크기 비교**: tiny / base / small로 재현율, 오경보, **처리 시간(GPU와 CPU)** 을 비교합니다. 파이에서 돌릴 수 있는지 판단하는 자료입니다.
3. **다른 녹음(학습용·시험용 87분)** 에도 단어 인식을 돌려서 TV 위급 장면, 노래 등에서 위급 단어가 나오는지 봅니다.
4. **소리 모델의 쓸모**: 단어로는 못 잡는 소리(신음, 비명, 쿵)를 AI Hub 시험 클립으로, 종류별 재현율과 C 오경보로 확인합니다.

**3번, 4번, 5번 항목의 출력**을 복사해서 알려 주세요.

## 0. 드라이브 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. 파일 찾기

In [ ]:
import glob, os, re, subprocess, pickle, time, csv, unicodedata
import numpy as np
import pandas as pd

def find(name):
    hits = glob.glob(f"/content/drive/MyDrive/**/{name}", recursive=True)
    assert hits, f"{name}을 못 찾았어요. 드라이브에 올렸는지 확인하세요."
    return hits[0]

OWN = find("6_own")
EMB = find("embeddings_v2.npz")
OUT_DIR = os.path.dirname(EMB)
files = sorted(glob.glob(f"{OWN}/*/*.m4a") + glob.glob(f"{OWN}/*/*.wav") + glob.glob(f"{OWN}/*/*.mp3"))
print("녹음 파일", len(files), "개 / 작업 폴더:", OUT_DIR)

## 2. 캐시와 말소리 구간 준비
07에서 만든 임베딩 캐시(`own_embeddings_v1.pkl`)를 쓰고, 오디오는 말소리 구간만 잘라 둡니다. (몇 분)

In [ ]:
FFMPEG = "ffmpeg"
SR, HOP, WIN = 16000, 7680, 15600
SEG = HOP * 600          # 288초씩 나눠서 처리 (프레임 격자는 이어짐)

def probe_channels(path):
    r = subprocess.run([FFMPEG, "-hide_banner", "-i", path], capture_output=True, text=True, errors="replace")
    m = re.search(r"Audio: .*?, \d+ Hz, ([^,\n]+)", r.stderr)
    return 1 if (m and m.group(1).strip().startswith(("mono", "1 channel"))) else 2

def load_mono(path):
    ch = probe_channels(path)
    raw = subprocess.run([FFMPEG, "-v", "error", "-i", path, "-vn", "-ac", str(ch), "-ar", str(SR), "-f", "s16le", "-"],
                         capture_output=True).stdout
    x = np.frombuffer(raw[: len(raw) // (2 * ch) * (2 * ch)], dtype=np.int16).reshape(-1, ch)
    if ch == 1:
        return x[:, 0].copy(), "모노", None
    a, b = x[::10, 0].astype(np.float64), x[::10, 1].astype(np.float64)
    corr = float(np.corrcoef(a, b)[0, 1]) if a.std() > 0 and b.std() > 0 else 1.0
    if corr < -0.3:
        return x[:, int(b.std() > a.std())].copy(), "한 채널(위상 반대)", corr
    return x.astype(np.float32).mean(axis=1).astype(np.int16), "두 채널 평균", corr

def embed_stream(x):
    embs, sp, sc, db, ts = [], [], [], [], []
    n = len(x)
    for start in range(0, n, SEG):
        seg = x[start:start + SEG + (WIN - HOP)]
        if len(seg) < WIN:
            break
        w = seg.astype(np.float32) / 32768
        scores, emb, _ = yamnet(tf.constant(w))
        scores, emb = scores.numpy(), emb.numpy()
        last = start + SEG + (WIN - HOP) >= n
        nb = len(w) // HOP
        bp = (w[: nb * HOP].reshape(nb, HOP) ** 2).mean(axis=1)
        fdb = 10 * np.log10(((bp[:-1] + bp[1:]) / 2 if nb > 1 else bp) + 1e-12)
        k = min(len(emb) if last else 600, len(emb), len(fdb))
        embs.append(emb[:k].astype(np.float16))
        sp.append(scores[:k, SPEECH].astype(np.float16))
        sc.append(scores[:k][:, SCREAM].max(axis=1).astype(np.float16))
        db.append(fdb[:k].astype(np.float32))
        ts.append(start / SR + np.arange(k) * 0.48)
    return tuple(np.concatenate(a) for a in (embs, sp, sc, db, ts))


import re, time
PKL = f"{OUT_DIR}/own_embeddings_v1.pkl"
assert os.path.exists(PKL), "own_embeddings_v1.pkl이 없어요. 07을 먼저 실행하세요."
own = pickle.load(open(PKL, "rb"))
nfc = lambda s: unicodedata.normalize("NFC", s)
for v in own.values():
    v["split"], v["name"] = nfc(v["split"]), nfc(v["name"])
E_files = [f for f, v in own.items() if "응급" in v["split"]]
C_files = [f for f, v in own.items() if ("장시간" in v["split"]) or ("연속" in v["name"]) or ("3시간" in v["name"])]
OTHER = [f for f in own if f not in E_files and f not in C_files]
assert E_files and C_files, "응급 소리/장시간 음성 녹음을 못 찾았어요."
hoursC = sum(own[f]["dur"] for f in C_files) / 3600
SPEECH_TH = 0.3

def speech_windows(v, gap):
    sp, t = v["speech"].astype(np.float32), v["t"]
    out = []
    for i in np.flatnonzero(sp > SPEECH_TH):
        if out and t[i] - t[out[-1][1]] <= gap:
            out[-1][1] = i
        else:
            out.append([i, i])
    return [(float(t[a]), float(t[b]) + 0.96) for a, b in out]

def make_windows(v, gap):
    out = []
    for k, (c0, c1) in enumerate(speech_windows(v, gap)):
        a, w1 = max(0.0, c0 - 0.5), min(v["dur"], c1 + 0.5)
        while True:
            b = min(a + 10.0, w1)
            out.append((k, a, b))
            if b >= w1 - 1e-6:
                break
            a += 9.0
    return out

WIN = {}
for f in E_files + C_files + OTHER:
    x, _, _ = load_mono(f)
    gap = 1.5 if f in E_files else 1.0
    WIN[f] = [(k, a, b, x[int(a * SR):int(b * SR)].astype(np.float32) / 32768) for k, a, b in make_windows(own[f], gap)]
    del x
n_cl = {f: len(speech_windows(own[f], 1.5)) for f in E_files}
n_E = sum(n_cl.values())
tot = sum(len(w) for w in WIN.values())
print(f"응급 발화 {n_E}개, C {hoursC:.2f}시간, 다른 녹음 {len(OTHER)}개 / 말소리 구간 합계 {tot}개 "
      f"({sum(b - a for w in WIN.values() for _, a, b, _ in w) / 60:.0f}분)")

SUF = r"(?:주|줘|달|요|세)"
STRICT = re.compile(r"(?:^|[\s,.!?])(?:도와\s*%s|살려\s*%s|구해\s*%s|사람\s*살려|119|일일구)" % (SUF, SUF, SUF))
LOOSE = re.compile(r"(?:^|[\s,.!?])(?:[도토또][와아]\s*%s|[살쌀]려\s*%s|구[해애]\s*%s|사람\s*살려|119|일일구)" % (SUF, SUF, SUF))

## 3. Whisper 크기별 평가 (tiny / base / small)
- **엄격 / 느슨**: 단어 규칙의 두 가지입니다. 느슨은 "토아주세요" 같은 철자 변형도 인정합니다.
- **E**: 응급 발화(두 기기 합계)의 재현율. **C**: 연속 3시간의 위급 단어 인식 횟수와 시간당 횟수. **다른 녹음**: 학습용·시험용 87분에서 인식된 횟수.
- **초/구간**: 구간 하나(몇 초짜리)를 처리하는 평균 시간. CPU는 20구간으로 잰 값(배치 1)이고, **파이4는 이보다 몇 배 느릴 것**으로 봐야 합니다.

In [ ]:
import torch, transformers
from transformers import pipeline
transformers.logging.set_verbosity_error()
dev = 0 if torch.cuda.is_available() else -1
print("런타임:", "GPU" if dev == 0 else "CPU (오래 걸려요)")
GEN = {"language": "ko", "task": "transcribe", "max_new_tokens": 48}
TXT = {}      # TXT[모델][파일] = 글자 목록
rows = []

def run(asr, wins):
    out = []
    for s in range(0, len(wins), 128):
        batch = [{"raw": arr, "sampling_rate": SR} for _, _, _, arr in wins[s:s + 128]]
        out += [r["text"] for r in asr(batch, batch_size=16, generate_kwargs=GEN)]
    return out

def metrics(texts, rx):
    e = sum(any(rx.search(tx) for (kk, _, _, _), tx in zip(WIN[f], texts[f]) if kk == k) for f in E_files for k in range(n_cl[f]))
    c = sum(bool(rx.search(tx)) for f in C_files for tx in texts[f])
    o = sum(bool(rx.search(tx)) for f in OTHER for tx in texts[f])
    return e, c, o

for name in ("openai/whisper-tiny", "openai/whisper-base", "openai/whisper-small"):
    short = name.split("-")[1]
    asr = pipeline("automatic-speech-recognition", model=name, device=dev, torch_dtype=torch.float16 if dev == 0 else torch.float32)
    t0 = time.time()
    TXT[short] = {f: run(asr, WIN[f]) for f in E_files + C_files + OTHER}
    sec_gpu = (time.time() - t0) / tot
    del asr
    if dev == 0:
        torch.cuda.empty_cache()
    cpu = pipeline("automatic-speech-recognition", model=name, device=-1, torch_dtype=torch.float32)
    sample = [w for w in WIN[C_files[0]] if 3 < w[2] - w[1] < 12][:20] or WIN[C_files[0]][:20]
    t0 = time.time()
    for w in sample:
        cpu({"raw": w[3], "sampling_rate": SR}, generate_kwargs=GEN)
    sec_cpu = (time.time() - t0) / max(len(sample), 1)
    del cpu
    for rx_name, rx in (("엄격", STRICT), ("느슨", LOOSE)):
        e, c, o = metrics(TXT[short], rx)
        rows.append({"모델": short, "단어규칙": rx_name, "E재현율": f"{e}/{n_E}", "C횟수": c, "C시간당": round(c / hoursC, 2),
                     "다른녹음횟수": o, "초/구간(GPU)": round(sec_gpu, 2), "초/구간(CPU)": round(sec_cpu, 2)})
    print(f"{short} 완료", flush=True)
pd.set_option("display.width", 200)
res_tbl = pd.DataFrame(rows)
print(res_tbl.to_string(index=False))

## 4. 무엇이 인식되고 놓쳤나
- **C·다른 녹음에서 위급 단어가 인식된 구간**(small, 느슨 규칙): 실제 위급이 아닌 말이었다면 오경보의 원인입니다.
- **응급 발화 중 인식하지 못한 것**: 모델 크기별로 어떤 글자로 들렸는지 봅니다.

In [ ]:
mmss = lambda s: f"{int(s) // 3600}:{int(s) % 3600 // 60:02d}:{int(s) % 60:02d}"
print("[위급 단어가 인식된 C·다른 녹음 구간: small, 느슨]")
for f in C_files + OTHER:
    for (k, a, b, _), tx in zip(WIN[f], TXT["small"][f]):
        if LOOSE.search(tx):
            print(f"  {own[f]['split']}/{own[f]['name'][:22]} {mmss(a)}  {tx.strip()[:50]}")
print("\n[응급 발화 중 놓친 것: 모델별 인식 글자 (느슨)]")
for short in TXT:
    for f in E_files:
        for k in range(n_cl[f]):
            ws = [(a, tx) for (kk, a, _, _), tx in zip(WIN[f], TXT[short][f]) if kk == k]
            if not any(LOOSE.search(tx) for _, tx in ws):
                print(f"  {short} {own[f]['name'][:18]} {mmss(ws[0][0])[2:]}  {' '.join(tx.strip() for _, tx in ws)[:40]}")

## 5. 소리 모델이 잡을 수 있는 위급 소리 (신음, 비명, 쿵)
AI Hub 시험 위급 클립을 **종류별**로 나눠서 재현율을 보고, 같은 설정의 C 오경보와 함께 봅니다. 신음·충격음 종류를 오경보 시간당 몇 회 수준에서 잡을 수 있는지가 핵심입니다.
(종류는 라벨의 세부 분류 이름으로 나눕니다: 신음·비명 / 넘어지는·떨어지는·구르는 소리 / 기침 / 그 외 말)

In [ ]:
from tensorflow import keras
d = np.load(EMB)
Xa, cidx = d["X"].astype(np.float32), d["clip_idx"]
df = pd.read_csv(find("dataset_v2.csv"), encoding="utf-8-sig")
y = (df["label"] == "urgent").astype(int).values
is_tr = (df["split"] == "train").values
starts = np.r_[0, np.flatnonzero(np.diff(cidx)) + 1]
ends = np.r_[starts[1:], len(cidx)]
vi = np.flatnonzero((~is_tr) & (y == 1))
cat = df["category"].astype(str).values[vi]
grp = np.select([pd.Series(cat).str.contains("신음|비명").values,
                 pd.Series(cat).str.contains("넘어지는|떨어지는|구르는|미끄러지는").values,
                 pd.Series(cat).str.contains("기침").values], ["신음·비명", "쿵·넘어짐", "기침"], "말")
print("Validation 위급 클립 종류별 개수:", {g: int((grp == g).sum()) for g in ["신음·비명", "쿵·넘어짐", "기침", "말"]})

def alarm_n(p, t, N, thr, cd=5.0):
    ok = (p >= thr).astype(int)
    if len(ok) < N:
        return 0
    run = np.convolve(ok, np.ones(N, int), "valid") == N
    n, last = 0, -1e9
    for i in np.flatnonzero(run) + (N - 1):
        if t[i] - last >= cd:
            n += 1; last = t[i]
    return n

def clip_sc(p, N):
    return np.array([np.lib.stride_tricks.sliding_window_view(p[starts[i]:ends[i]], N).min(axis=1).max()
                     if ends[i] - starts[i] >= N else 0.0 for i in vi])

rows = []
for label, fn in (("03b", "classifier_mlp_v2.keras"), ("04", "classifier_mlp_v3_own.keras"), ("05", "classifier_mlp_v4_own_all.keras")):
    try:
        m = keras.models.load_model(find(fn))
    except AssertionError as e:
        print("건너뜀:", e); continue
    pa = m.predict(Xa, batch_size=4096, verbose=0).ravel()
    pc = {f: m.predict(own[f]["emb"].astype(np.float32), batch_size=4096, verbose=0).ravel() for f in C_files}
    for N in (1, 2):
        sc = clip_sc(pa, N)
        for thr in (0.5, 0.9, 0.95):
            r = {"모델": label, "N": N, "임계값": thr,
                 "C오경보/시간": round(sum(alarm_n(pc[f], own[f]["t"], N, thr) for f in C_files) / hoursC, 1)}
            for g in ["신음·비명", "쿵·넘어짐", "기침", "말"]:
                r[g] = round(float((sc[grp == g] >= thr).mean()), 3) if (grp == g).any() else None
            rows.append(r)
print(pd.DataFrame(rows).to_string(index=False))

## 6. 결과 저장

In [ ]:
res_tbl.to_csv(f"{OUT_DIR}/result_whisper_sizes.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(rows).to_csv(f"{OUT_DIR}/result_sound_path.csv", index=False, encoding="utf-8-sig")
print("저장 완료:", OUT_DIR)